In [1]:
import pandas as pd
import numpy as np
from indices import *
import os

In [2]:
df = pd.read_csv("selected_features_per_scenario.csv")
df.head()

,Code,Method,Feature class,Set size,"Time, s","Memory peak, mb",Selected features ids,IoU train,IoU test,F1 train,F1 test,Selected features names,* - removed duplicates and constants
0,MRMR_2,MRMR,NORMP,55*,17.5,206.5,"13, 30, 54",0.558,0.551,0.716,0.711,"NORMP(B4, B3) NORMP(B9, B4) NORMP(B12, B6)","long - 100 generations, 1000 - polulation"
1,MRMR_3,MRMR,HueSimp,605*,155.9,1691.3,"145, 164, 724",0.561,0.562,0.718,0.720,"HueSimp(B4, B4, B3) HueSimp(B12, B5, B3) HueSi...","short - 100 generations, 250 - population"
2,MRMR_4,MRMR,NORMP4,3025*,763.9,5454.2,"134, 6588, 13214",0.537,0.594,0.699,0.746,"NORMP4(B4, B3, B3, B2), NORMP4(B12, B6, B12, B...",NaN
3,PL_2,Proposed (long),NORMP,121,15.2,16.2,"13, 113, 97",0.567,0.599,0.723,0.749,"NORMP(B4, B3), NORMP(B5, B12), NORMP(B11, B9)",NaN
4,PS_2,Proposed (short),NORMP,121,4.4,14.7,"23, 118, 113",0.551,0.618,0.710,0.764,"NORMP(B3, B4), NORMP(B9, B12), NORMP(B5, B12)",NaN


In [3]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

data = np.concatenate([data[0], data[1]], axis=1)
data.shape

(12, 38978)

In [4]:
with open("bulk_gen_template.yaml", "r") as template:
    bulk_config = template.read()

bands_range = list(range(1, 12))
for i, row in df.iterrows():
    scenario_name = row["Code"]
    feature_class = row["Feature class"]
    str_selected_features = row["Selected features ids"]

    if str_selected_features == "-":
        continue

    feature_ids = [int(v) for v in str_selected_features.split(", ")]
    if len(feature_ids) < 3:
        continue

    print(scenario_name, feature_class, str_selected_features)

    with open("vv2rgb_gen_template.py", "r") as template:
        code_text = template.read()

    code_text = code_text.replace("___FEATURECLASS", feature_class)

    if feature_class == "NORMP":
        encoder = IndicesClassEncoderEq([NORMP], bands_range)
    elif feature_class == "HueSimp":
        encoder = IndicesClassEncoderEq([HueSimp], bands_range)
    elif feature_class == "NORMP4":
        encoder = IndicesClassEncoderEq([NORMP4], bands_range)

    for j in range(3):
        feature_id = feature_ids[j]

        feature = encoder.getIndex(feature_id)
        values = feature.getValue(data)
        q_min = np.quantile(values, 0.05)
        q_max = np.quantile(values, 0.95)
        print(f"Feature {j}: Q5:", q_min, "Q95", q_max)

        code_text = code_text.replace(f"___FEATURE_{j}_MINMAX", f"[{q_min}, {q_max}]")
        code_text = code_text.replace(f"___FEATURE_{j}", str(feature_id))

    save_path = os.path.join("training\\vv2rgb", scenario_name + ".py")
    with open(save_path, "w+") as file:
        file.write(code_text)

    with open("config_gen_template.yaml", "r") as template:
        code_text = template.read()

    code_text = code_text.replace("___SCENARIO_NAME", scenario_name)
    save_path = os.path.join("training\configs\generated", scenario_name + ".yaml")
    with open(save_path, "w+") as file:
        file.write(code_text)

    bulk_config += f"\n  - generated\\{scenario_name}.yaml"

    print()

save_path = os.path.join("training\\bulks", "bulk_gen.yaml")
with open(save_path, "w+") as file:
    file.write(bulk_config)

MRMR_2 NORMP 13, 30, 54
Feature 0: Q5: -0.3045226812229239 Q95 0.10015644019435427
Feature 1: Q5: 0.6198521858327408 Q95 0.8922155746789795
Feature 2: Q5: -0.6560889572250587 Q95 -0.19370390259716894

MRMR_3 HueSimp 145, 164, 724
Feature 0: Q5: 4.90000942945934e-07 Q95 0.0001988100073957444
Feature 1: Q5: -0.00025439994701594097 Q95 0.0019416993488381067
Feature 2: Q5: -0.0039511999869823455 Q95 0.012856470103411679

MRMR_4 NORMP4 134, 6588, 13214
Feature 0: Q5: -0.31173384247945846 Q95 0.1322701702045459
Feature 1: Q5: -0.6560889572250587 Q95 -0.19370390259716894
Feature 2: Q5: 0.1012658159203912 Q95 0.3126548570292646

PL_2 NORMP 13, 113, 97
Feature 0: Q5: -0.3045226812229239 Q95 0.10015644019435427
Feature 1: Q5: -0.10797463458503789 Q95 0.20709187579758495
Feature 2: Q5: -0.5313874259399313 Q95 -0.10993649296520587

PS_2 NORMP 23, 118, 113
Feature 0: Q5: -0.10015644019435425 Q95 0.304522681222924
Feature 1: Q5: 0.3838383885164499 Q95 0.7626678241041849
Feature 2: Q5: -0.10797463458